# Transfer Learning for Image Classification

## Introduction

In the previous notebook, we explored transfer learning for NLP using pre-trained word embeddings (GloVe). In this notebook, we apply the same concept to **image classification** using pre-trained Convolutional Neural Networks (CNNs).

### Why Transfer Learning for Images?

Pre-trained models like VGG16, ResNet50, and MobileNetV2 have learned powerful visual features from **ImageNet** (1.2 million images, 1000 classes):
- **Early layers**: Edges, colors, textures (universal features)
- **Middle layers**: Patterns, shapes, object parts
- **Deep layers**: High-level concepts specific to ImageNet classes

We can leverage these learned features for our own tasks!

### Learning Objectives
- Understand transfer learning strategies for images
- Use pre-trained models from `tf.keras.applications`
- Compare Feature Extraction vs Fine-tuning
- Apply multiple models to Fashion MNIST
- Understand when each approach works best

### Prerequisites
- Completed Module 01-06 and CNN Fundamentals
- Understanding of convolution operations
- Familiarity with model training and evaluation

---

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import time

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Load Fashion MNIST
# Fashion MNIST has 10 classes of clothing items
(X_train_orig, y_train), (X_test_orig, y_test) = fashion_mnist.load_data()

# Class names for Fashion MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training data shape: {X_train_orig.shape}")
print(f"Test data shape: {X_test_orig.shape}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_orig[i], cmap='gray')
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.suptitle('Fashion MNIST Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. The Challenge: Adapting Fashion MNIST for Pre-trained Models

### The Problem
- **Fashion MNIST**: 28×28 grayscale images
- **Pre-trained models (ImageNet)**: Expect 224×224 (or similar) RGB images

### Solutions
1. **Resize** images from 28×28 to the required input size
2. **Convert grayscale to RGB** by repeating the channel 3 times
3. **Apply model-specific preprocessing** (each model has its own normalization)

### Important Considerations
- Fashion MNIST images are simple compared to ImageNet
- Pre-trained features may be "overkill" for this dataset
- This is a learning exercise - real-world transfer learning shines with complex images

In [ ]:
def preprocess_for_transfer_learning(X, target_size=(96, 96)):
    """
    Preprocess Fashion MNIST images for pre-trained models.
    
    Args:
        X: Input images (N, 28, 28)
        target_size: Target size for resizing (height, width)
    
    Returns:
        Preprocessed images (N, target_size[0], target_size[1], 3)
    """
    # Normalize to 0-1 first
    X = X.astype('float32') / 255.0
    
    # Add channel dimension: (N, 28, 28) -> (N, 28, 28, 1)
    X = np.expand_dims(X, axis=-1)
    
    # Convert grayscale to RGB: (N, 28, 28, 1) -> (N, 28, 28, 3)
    X = np.repeat(X, 3, axis=-1)
    
    # Resize to target size using tf.image.resize
    X = tf.image.resize(X, target_size).numpy()
    
    return X

# We'll use 96x96 as a compromise between speed and quality
# (224x224 would be better but slower for this demo)
TARGET_SIZE = (96, 96)

print("Preprocessing images (this may take a moment)...")
X_train = preprocess_for_transfer_learning(X_train_orig, TARGET_SIZE)
X_test = preprocess_for_transfer_learning(X_test_orig, TARGET_SIZE)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Preprocessed training shape: {X_train.shape}")
print(f"Preprocessed test shape: {X_test.shape}")

In [ ]:
# Visualize preprocessed images
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

# Original
axes[0].imshow(X_train_orig[0], cmap='gray')
axes[0].set_title('Original (28x28, grayscale)')

# Resized grayscale
axes[1].imshow(X_train[0][:,:,0], cmap='gray')
axes[1].set_title(f'Resized ({TARGET_SIZE[0]}x{TARGET_SIZE[1]})')

# RGB version
axes[2].imshow(X_train[0])
axes[2].set_title('RGB conversion')

# Show RGB channels
axes[3].imshow(np.hstack([X_train[0][:,:,i] for i in range(3)]), cmap='gray')
axes[3].set_title('R, G, B channels (identical)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Transfer Learning Strategies

### Strategy 1: Feature Extraction
- **Freeze** all pre-trained layers (don't update their weights)
- Add new classification layers on top
- Only train the new layers
- **Pros**: Fast, works with small datasets
- **Cons**: Can't adapt features to new domain

### Strategy 2: Fine-tuning
- Start with pre-trained weights
- **Unfreeze** some or all layers
- Train the entire model with a small learning rate
- **Pros**: Better adaptation to new domain
- **Cons**: Risk of overfitting, needs more data

### When to Use Each

| Scenario | Strategy |
|----------|----------|
| Small dataset, similar to ImageNet | Feature Extraction |
| Large dataset, similar to ImageNet | Fine-tuning |
| Small dataset, different from ImageNet | Feature Extraction + Data Aug |
| Large dataset, different from ImageNet | Fine-tuning (gradual unfreezing) |

## 4. Available Pre-trained Models in TensorFlow

TensorFlow provides many pre-trained models via `tf.keras.applications`:

| Model | Parameters | Top-1 Accuracy | Speed |
|-------|------------|----------------|-------|
| **MobileNetV2** | 3.4M | 71.3% | Very Fast |
| **VGG16** | 138M | 71.3% | Slow |
| **ResNet50** | 25.6M | 74.9% | Medium |
| **InceptionV3** | 23.8M | 77.9% | Medium |
| **EfficientNetB0** | 5.3M | 77.1% | Fast |

For this demo, we'll compare:
1. **MobileNetV2** - Lightweight, fast, good for mobile/edge devices
2. **VGG16** - Classic architecture, easy to understand
3. **ResNet50** - Deep residual network, state-of-the-art

## 5. Feature Extraction with MobileNetV2

In [ ]:
def create_feature_extraction_model(base_model_fn, input_shape, num_classes, model_name):
    """
    Create a feature extraction model using a pre-trained base.
    
    Args:
        base_model_fn: Function to create base model (e.g., applications.MobileNetV2)
        input_shape: Input shape for images
        num_classes: Number of output classes
        model_name: Name for the model
    
    Returns:
        Compiled Keras model
    """
    # Load pre-trained model WITHOUT the top classification layer
    base_model = base_model_fn(
        weights='imagenet',      # Use ImageNet pre-trained weights
        include_top=False,       # Don't include the final classification layers
        input_shape=input_shape  # Our input shape
    )
    
    # Freeze the base model - we won't train these layers
    base_model.trainable = False
    
    # Create new model with our classification head
    inputs = keras.Input(shape=input_shape)
    
    # Apply base model's preprocessing
    # Each model has its own preprocessing function
    x = inputs
    
    # Get features from base model
    x = base_model(x, training=False)
    
    # Add our classification layers
    x = layers.GlobalAveragePooling2D()(x)  # Convert feature maps to vector
    x = layers.Dropout(0.3)(x)               # Regularization
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name=model_name)
    
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model, base_model

In [ ]:
# Create MobileNetV2 feature extraction model
input_shape = (*TARGET_SIZE, 3)

print("Creating MobileNetV2 feature extraction model...")
model_mobilenet, base_mobilenet = create_feature_extraction_model(
    applications.MobileNetV2,
    input_shape,
    num_classes=10,
    model_name='MobileNetV2_FeatureExtraction'
)

# Show model summary
print(f"\nTotal parameters: {model_mobilenet.count_params():,}")
print(f"Trainable parameters: {sum(tf.keras.backend.count_params(w) for w in model_mobilenet.trainable_weights):,}")
print(f"Non-trainable (frozen) parameters: {sum(tf.keras.backend.count_params(w) for w in model_mobilenet.non_trainable_weights):,}")

In [ ]:
# Train MobileNetV2 (Feature Extraction)
# Using a smaller subset for faster training in this demo
TRAIN_SAMPLES = 10000  # Use 10k samples for faster demo
TEST_SAMPLES = 2000

print(f"Training on {TRAIN_SAMPLES} samples...")
start_time = time.time()

history_mobilenet = model_mobilenet.fit(
    X_train[:TRAIN_SAMPLES], y_train_cat[:TRAIN_SAMPLES],
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

mobilenet_time = time.time() - start_time
print(f"\nTraining time: {mobilenet_time:.1f} seconds")

In [ ]:
# Evaluate MobileNetV2
mobilenet_loss, mobilenet_acc = model_mobilenet.evaluate(
    X_test[:TEST_SAMPLES], y_test_cat[:TEST_SAMPLES], verbose=0
)
print(f"MobileNetV2 Test Accuracy: {mobilenet_acc:.4f}")

## 6. Feature Extraction with VGG16

In [ ]:
# Create VGG16 feature extraction model
print("Creating VGG16 feature extraction model...")
model_vgg16, base_vgg16 = create_feature_extraction_model(
    applications.VGG16,
    input_shape,
    num_classes=10,
    model_name='VGG16_FeatureExtraction'
)

print(f"\nTotal parameters: {model_vgg16.count_params():,}")
print(f"Trainable parameters: {sum(tf.keras.backend.count_params(w) for w in model_vgg16.trainable_weights):,}")

In [ ]:
# Train VGG16
print(f"Training VGG16 on {TRAIN_SAMPLES} samples...")
start_time = time.time()

history_vgg16 = model_vgg16.fit(
    X_train[:TRAIN_SAMPLES], y_train_cat[:TRAIN_SAMPLES],
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

vgg16_time = time.time() - start_time
print(f"\nTraining time: {vgg16_time:.1f} seconds")

In [ ]:
# Evaluate VGG16
vgg16_loss, vgg16_acc = model_vgg16.evaluate(
    X_test[:TEST_SAMPLES], y_test_cat[:TEST_SAMPLES], verbose=0
)
print(f"VGG16 Test Accuracy: {vgg16_acc:.4f}")

## 7. Feature Extraction with ResNet50

In [ ]:
# Create ResNet50 feature extraction model
print("Creating ResNet50 feature extraction model...")
model_resnet50, base_resnet50 = create_feature_extraction_model(
    applications.ResNet50,
    input_shape,
    num_classes=10,
    model_name='ResNet50_FeatureExtraction'
)

print(f"\nTotal parameters: {model_resnet50.count_params():,}")
print(f"Trainable parameters: {sum(tf.keras.backend.count_params(w) for w in model_resnet50.trainable_weights):,}")

In [ ]:
# Train ResNet50
print(f"Training ResNet50 on {TRAIN_SAMPLES} samples...")
start_time = time.time()

history_resnet50 = model_resnet50.fit(
    X_train[:TRAIN_SAMPLES], y_train_cat[:TRAIN_SAMPLES],
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

resnet50_time = time.time() - start_time
print(f"\nTraining time: {resnet50_time:.1f} seconds")

In [ ]:
# Evaluate ResNet50
resnet50_loss, resnet50_acc = model_resnet50.evaluate(
    X_test[:TEST_SAMPLES], y_test_cat[:TEST_SAMPLES], verbose=0
)
print(f"ResNet50 Test Accuracy: {resnet50_acc:.4f}")

## 8. Comparing All Models

In [ ]:
# Compare results
results = {
    'Model': ['MobileNetV2', 'VGG16', 'ResNet50'],
    'Test Accuracy': [mobilenet_acc, vgg16_acc, resnet50_acc],
    'Training Time (s)': [mobilenet_time, vgg16_time, resnet50_time],
    'Total Parameters': [
        model_mobilenet.count_params(),
        model_vgg16.count_params(),
        model_resnet50.count_params()
    ]
}

print("\n" + "="*70)
print("MODEL COMPARISON - Feature Extraction on Fashion MNIST")
print("="*70)
print(f"{'Model':<15} {'Accuracy':<12} {'Time (s)':<12} {'Parameters':<15}")
print("-"*70)
for i in range(3):
    print(f"{results['Model'][i]:<15} {results['Test Accuracy'][i]:.4f}       {results['Training Time (s)'][i]:<12.1f} {results['Total Parameters'][i]:,}")
print("="*70)

In [ ]:
# Visualize training history comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].plot(history_mobilenet.history['val_accuracy'], label='MobileNetV2', linewidth=2)
axes[0].plot(history_vgg16.history['val_accuracy'], label='VGG16', linewidth=2)
axes[0].plot(history_resnet50.history['val_accuracy'], label='ResNet50', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Validation Accuracy During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss comparison
axes[1].plot(history_mobilenet.history['val_loss'], label='MobileNetV2', linewidth=2)
axes[1].plot(history_vgg16.history['val_loss'], label='VGG16', linewidth=2)
axes[1].plot(history_resnet50.history['val_loss'], label='ResNet50', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Loss')
axes[1].set_title('Validation Loss During Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

models = results['Model']
colors = ['#2ecc71', '#3498db', '#e74c3c']

# Accuracy
axes[0].bar(models, results['Test Accuracy'], color=colors)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Test Accuracy')
axes[0].set_ylim(0.7, 0.95)
for i, v in enumerate(results['Test Accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

# Training Time
axes[1].bar(models, results['Training Time (s)'], color=colors)
axes[1].set_ylabel('Seconds')
axes[1].set_title('Training Time')
for i, v in enumerate(results['Training Time (s)']):
    axes[1].text(i, v + 1, f'{v:.0f}s', ha='center')

# Parameters (in millions)
params_millions = [p/1e6 for p in results['Total Parameters']]
axes[2].bar(models, params_millions, color=colors)
axes[2].set_ylabel('Parameters (millions)')
axes[2].set_title('Model Size')
for i, v in enumerate(params_millions):
    axes[2].text(i, v + 1, f'{v:.1f}M', ha='center')

plt.tight_layout()
plt.show()

## 9. Fine-tuning: Unfreezing Top Layers

Now let's try fine-tuning by unfreezing some of the top layers of the pre-trained model. This allows the model to adapt the learned features to our specific task.

In [ ]:
def create_fine_tuning_model(base_model_fn, input_shape, num_classes, unfreeze_layers=20):
    """
    Create a model for fine-tuning with some layers unfrozen.
    
    Args:
        base_model_fn: Function to create base model
        input_shape: Input shape for images
        num_classes: Number of output classes
        unfreeze_layers: Number of top layers to unfreeze
    
    Returns:
        Compiled Keras model
    """
    base_model = base_model_fn(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze all layers first
    base_model.trainable = True
    
    # Unfreeze the top N layers
    for layer in base_model.layers[:-unfreeze_layers]:
        layer.trainable = False
    
    # Count trainable layers
    trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
    print(f"Unfroze {trainable_count} layers out of {len(base_model.layers)}")
    
    # Build the model
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=True)  # training=True for fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    
    # Use a smaller learning rate for fine-tuning
    # This is CRITICAL - large learning rates will destroy pre-trained weights
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # Much smaller!
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [ ]:
# Create fine-tuned MobileNetV2
print("Creating fine-tuned MobileNetV2...")
model_mobilenet_ft = create_fine_tuning_model(
    applications.MobileNetV2,
    input_shape,
    num_classes=10,
    unfreeze_layers=30  # Unfreeze top 30 layers
)

trainable_params = sum(tf.keras.backend.count_params(w) for w in model_mobilenet_ft.trainable_weights)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Train fine-tuned model
print(f"Fine-tuning MobileNetV2 on {TRAIN_SAMPLES} samples...")
start_time = time.time()

history_mobilenet_ft = model_mobilenet_ft.fit(
    X_train[:TRAIN_SAMPLES], y_train_cat[:TRAIN_SAMPLES],
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

mobilenet_ft_time = time.time() - start_time
print(f"\nFine-tuning time: {mobilenet_ft_time:.1f} seconds")

In [ ]:
# Evaluate fine-tuned model
mobilenet_ft_loss, mobilenet_ft_acc = model_mobilenet_ft.evaluate(
    X_test[:TEST_SAMPLES], y_test_cat[:TEST_SAMPLES], verbose=0
)
print(f"\nMobileNetV2 Fine-tuned Test Accuracy: {mobilenet_ft_acc:.4f}")
print(f"MobileNetV2 Feature Extraction Test Accuracy: {mobilenet_acc:.4f}")
print(f"\nImprovement: {(mobilenet_ft_acc - mobilenet_acc)*100:.2f}%")

In [ ]:
# Compare Feature Extraction vs Fine-tuning
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_mobilenet.history['val_accuracy'], 'b-', label='MobileNetV2 (Feature Extraction)', linewidth=2)
ax.plot(history_mobilenet_ft.history['val_accuracy'], 'r-', label='MobileNetV2 (Fine-tuned)', linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Feature Extraction vs Fine-tuning')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Visualizing What the Model Learned

In [ ]:
# Make predictions and show some examples
predictions = model_mobilenet.predict(X_test[:20], verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for i, ax in enumerate(axes.flat):
    # Show original image (not resized)
    ax.imshow(X_test_orig[i], cmap='gray')
    
    true_label = class_names[y_test[i]]
    pred_label = class_names[predicted_classes[i]]
    confidence = predictions[i][predicted_classes[i]]
    
    color = 'green' if predicted_classes[i] == y_test[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.2f})', 
                 color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Model Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Best Practices for Transfer Learning

### Data Preprocessing
1. **Always match the input format** expected by the pre-trained model
2. **Use the model's preprocessing function** when available
3. **Resize images appropriately** - bigger isn't always better

### Feature Extraction
1. **Start with feature extraction** before trying fine-tuning
2. **Add regularization** (Dropout, L2) to the new layers
3. **Use GlobalAveragePooling2D** instead of Flatten for better generalization

### Fine-tuning
1. **Use a very small learning rate** (10x-100x smaller than normal)
2. **Unfreeze layers gradually** - start from the top
3. **Use early stopping** to prevent overfitting
4. **Consider unfreezing BatchNorm layers** carefully (they can destabilize training)

### Model Selection
| Use Case | Recommended Model |
|----------|-------------------|
| Mobile/Edge deployment | MobileNetV2/V3, EfficientNet-Lite |
| Best accuracy (no size limit) | EfficientNetB7, ConvNeXt |
| Good balance | ResNet50, EfficientNetB0 |
| Legacy systems | VGG16/19 |

## 12. Summary

### What We Learned

1. **Transfer Learning Concept**: Reuse knowledge from models trained on large datasets

2. **Two Strategies**:
   - **Feature Extraction**: Freeze pre-trained layers, only train new classifier
   - **Fine-tuning**: Unfreeze some layers, train with small learning rate

3. **Pre-trained Models**: MobileNetV2, VGG16, ResNet50 each have trade-offs

4. **Key Considerations**:
   - Input preprocessing must match training data format
   - Fine-tuning requires careful learning rate selection
   - More parameters != better results

### Comparison with NLP Transfer Learning

| Aspect | NLP (Previous Notebook) | Images (This Notebook) |
|--------|------------------------|------------------------|
| Pre-trained resource | Word embeddings (GloVe) | CNN weights (ImageNet) |
| What's transferred | Word relationships | Visual features |
| Preprocessing | Tokenization | Resize + RGB conversion |
| Common models | GloVe, Word2Vec, BERT | ResNet, VGG, MobileNet |

### Next Steps
- Try with real-world images (not Fashion MNIST)
- Experiment with data augmentation
- Explore EfficientNet family
- Learn about Vision Transformers (ViT)

## 13. Exercises

### Exercise 1: Try EfficientNetB0
Create a feature extraction model using `applications.EfficientNetB0`. Compare its accuracy, speed, and size with the models we tested.

### Exercise 2: Data Augmentation
Add data augmentation (rotation, flipping, zoom) to the training pipeline. Does it improve accuracy?

### Exercise 3: Different Datasets
Try the same approach on CIFAR-10 (32x32 color images, 10 classes). How do the results compare?

### Exercise 4: Gradual Unfreezing
Implement a training schedule where you:
1. Train only the classifier head (5 epochs)
2. Unfreeze top 20 layers and train (5 epochs)
3. Unfreeze more layers and train (5 epochs)

Compare with unfreezing all at once.

In [ ]:
# Exercise 1: Your code here - Try EfficientNetB0
# Hint: Use applications.EfficientNetB0

In [ ]:
# Exercise 2: Your code here - Data Augmentation
# Hint: Use tf.keras.layers.RandomFlip, RandomRotation, etc.

---

## Exercise Solutions

In [ ]:
# Solution: Exercise 1 - EfficientNetB0
print("Creating EfficientNetB0 feature extraction model...")

model_efficientnet, base_efficientnet = create_feature_extraction_model(
    applications.EfficientNetB0,
    input_shape,
    num_classes=10,
    model_name='EfficientNetB0_FeatureExtraction'
)

print(f"Total parameters: {model_efficientnet.count_params():,}")

# Train (uncomment to run)
# history_efficientnet = model_efficientnet.fit(
#     X_train[:TRAIN_SAMPLES], y_train_cat[:TRAIN_SAMPLES],
#     epochs=10, batch_size=32, validation_split=0.2, verbose=1
# )
# efficientnet_loss, efficientnet_acc = model_efficientnet.evaluate(
#     X_test[:TEST_SAMPLES], y_test_cat[:TEST_SAMPLES], verbose=0
# )
# print(f"EfficientNetB0 Test Accuracy: {efficientnet_acc:.4f}")

In [ ]:
# Solution: Exercise 2 - Data Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

# Create model with augmentation
base_model = applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=input_shape
)
base_model.trainable = False

inputs = keras.Input(shape=input_shape)
x = data_augmentation(inputs)  # Apply augmentation
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model_with_aug = keras.Model(inputs, outputs)
model_with_aug.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Model with data augmentation created!")
print("Data augmentation layers will apply random transformations during training.")